In [2]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

/opt/anaconda3/envs/py313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Perplexity (PPL) Evaluation

Perplexity measures how surprised a language model is by a given text. A lower PPL means the model assigns a higher probability to the text, so the text is considered more fluent and natural.

In this example, we compare two candidate responses and calculate the perplexity for each one using GPT-2. Lower perplexity generally indicates a better fit to the model's learned language distribution.

In [4]:
model = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer= GPT2Tokenizer.from_pretrained('gpt2')
model.eval()

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4039.28it/s]


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
# Example text candidates for perplexity comparison
responses = [
    "Tht capital of France is Paris.",
    "Frances capital is Paris the",
]

# Note:
# - The first sentence has a typo ("Tht"), so it should be slightly less likely under the model.
# - The second sentence is grammatically awkward and less coherent, so it should also score worse.
# We will compute perplexity to quantify this difference.

In [11]:
def compute_ppl(text):
    encodings = tokenizer(text, return_tensors='pt')
    input_ids = encodings.input_ids

    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
        loss = outputs.loss
    ppl = torch.exp(loss)
    return ppl.item()

In [ ]:
# Evaluate each response
for i, response in enumerate(responses):
    ppl = compute_ppl(response)
    print(f"Response {i+1}: {response}")
    print(f"Perplexity: {ppl: .2f}\n")

# Interpretation:
# A lower perplexity means the model finds the sentence more probable.
# In practice, the more fluent and grammatically consistent response should have the lower PPL.

Response 1: Tht capital of France is Paris.
Perplexity:  158.120575

Response 2: Frances capital is Paris the
Perplexity:  11413.598633



## Final conclusion

Perplexity is a useful way to compare how well a language model predicts different candidate outputs. In this example, we expect the more natural and coherent sentence to receive a lower perplexity score, because the model is more confident in predicting it.

This makes PPL a practical metric for evaluating generation quality when the goal is to rank candidate responses by fluency and likelihood. However, it is not the only metric to use: sentence-level grammar, factual correctness, and task-specific quality should also be considered alongside perplexity.